In [2]:
import sys, os
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)


In [ ]:
#affichage de la donnée d'entré
import numpy as np

data   = np.load('dataset/preprocessed_dataset.npz', allow_pickle=True)
hrtf   = data['hrtf']   # [N_sujets, 121, 129, 2]

sample = hrtf[0]        # [121, 129, 2] — un sujet
print(f'Shape : {sample.shape}')
print(sample)


Shape : (121, 129, 2)
[[[-2.53036022e+00 -2.52523661e+00]
  [-2.64581466e+00 -2.64486146e+00]
  [-2.56272006e+00 -2.54622293e+00]
  ...
  [-1.03678188e+01 -1.09438591e+01]
  [-1.07942772e+01 -1.12170486e+01]
  [-1.13991833e+01 -1.18237953e+01]]

 [[-4.28027213e-01  5.12623608e-01]
  [-4.43924755e-01  5.33664703e-01]
  [-7.70168841e-01  8.31963956e-01]
  ...
  [-6.10110474e+00  6.38195992e+00]
  [-6.06071329e+00  6.28240490e+00]
  [-6.00076103e+00  6.22876501e+00]]

 [[ 6.49911165e-02  6.48414865e-02]
  [ 5.90071641e-02  5.97212426e-02]
  [-1.28640179e-02 -2.61356607e-02]
  ...
  [ 2.42591038e-01 -3.18171769e-01]
  [ 2.72735894e-01 -2.67515063e-01]
  [ 3.73694062e-01 -1.64959833e-01]]

 ...

 [[ 5.99923432e-02  6.01251908e-02]
  [ 5.93562424e-02  6.08114935e-02]
  [ 5.67747653e-02  5.57995848e-02]
  ...
  [ 6.30335063e-02  3.42772976e-02]
  [ 7.79733658e-02  6.73745275e-02]
  [ 8.35511908e-02  7.34660104e-02]]

 [[ 5.91900498e-02  5.63728921e-02]
  [ 5.94165213e-02  5.64095825e-02]
  [ 

In [4]:
from src.models import HRTFEncoder

enc_hrtf = HRTFEncoder(n_sh=121, n_freqs=129, embedding_dim=128)
enc_hrtf.summary()


Model: "HRTFEncoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hrtf (InputLayer)               │ (None, 121, 129, 2)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 121, 258)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 121, 32)        │         8,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3872)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       991,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (UnitNormalization)   │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,032,672 (3.94 MB)

 Trainable params: 1,032,672 (3.94 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import numpy as np
import tensorflow as tf

dummy = np.random.rand(4, 121, 129, 2).astype('float32')

z = enc_hrtf.model(dummy, training=False)
print(f'Shape sortie : {z.shape}')                        # → (4, 128)
print(f'Normes L2    : {tf.norm(z, axis=-1).numpy()}')   # → [1. 1. 1. 1.]


Shape sortie : (4, 128)
Normes L2    : [0.99999994 0.99999994 0.99999994 1.        ]
